In [1]:
!pip install mplsoccer

  Using cached requests-2.34.2-py3-none-any.whl.metadata (4.8 kB)
  Using cached seaborn-0.13.2-py3-none-any.whl.metadata (5.4 kB)
  Using cached contourpy-1.3.3-cp313-cp313-win_amd64.whl.metadata (5.5 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached pyparsing-3.3.2-py3-none-any.whl.metadata (5.8 kB)
  Using cached tzdata-2026.3-py2.py3-none-any.whl.metadata (1.4 kB)
  Using cached charset_normalizer-3.5.1-cp313-cp313-win_amd64.whl.metadata (46 kB)
  Using cached idna-3.19-py3-none-any.whl.metadata (9.2 kB)
  Using cached urllib3-2.7.0-py3-none-any.whl.metadata (6.9 kB)
  Using cached certifi-2026.7.22-py3-none-any.whl.metadata (2.5 kB)
   ---------------------------------------- 0.0/9.3 MB ? eta -:--:--
   -- ------------------------------------- 0.5/9.3 MB 3.3 MB/s eta 0:00:03
   ----- ---------------------------------- 1.3/9.3 MB 3.7 MB/s eta 0:00:03
   -------- ------------------------------- 2.1/9.3 MB 3.8 MB/s eta 0:00:02
   ------------ --------

In [3]:
!pip install bs4

  Using cached beautifulsoup4-4.15.0-py3-none-any.whl.metadata (3.8 kB)
  Using cached soupsieve-2.9.2-py3-none-any.whl.metadata (4.6 kB)
Using cached beautifulsoup4-4.15.0-py3-none-any.whl (109 kB)
Using cached soupsieve-2.9.2-py3-none-any.whl (37 kB)

   ---------------------------------------- 0/3 [soupsieve]
   ------------- -------------------------- 1/3 [beautifulsoup4]
   ------------- -------------------------- 1/3 [beautifulsoup4]
   ---------------------------------------- 3/3 [bs4]



In [5]:
!pip install plotly

   ---------------------------------------- 0.0/9.1 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.1 MB ? eta -:--:--
   -- ------------------------------------- 0.5/9.1 MB 2.3 MB/s eta 0:00:04
   -- ------------------------------------- 0.5/9.1 MB 2.3 MB/s eta 0:00:04
   -- ------------------------------------- 0.5/9.1 MB 2.3 MB/s eta 0:00:04
   ----- ---------------------------------- 1.3/9.1 MB 1.5 MB/s eta 0:00:06
   --------- ------------------------------ 2.1/9.1 MB 1.9 MB/s eta 0:00:04
   ------------ --------------------------- 2.9/9.1 MB 2.2 MB/s eta 0:00:03
   ---------------- ----------------------- 3.7/9.1 MB 2.4 MB/s eta 0:00:03
   ------------------- -------------------- 4.5/9.1 MB 2.6 MB/s eta 0:00:02
   ----------------------- ---------------- 5.2/9.1 MB 2.8 MB/s eta 0:00:02
   -------------------------- ------------- 6.0/9.1 MB 2.8 MB/s eta 0:00:02
   ------------------------------ --------- 6.8/9.1 MB 2.9 MB/s eta 0:00:01
   -----------------------

In [16]:
import requests
from bs4 import BeautifulSoup
import json
import pandas as pd
from mplsoccer import Pitch, VerticalPitch
import matplotlib.pyplot as plt
from pandas import json_normalize
import matplotlib.dates as mdates
from mplsoccer import Pitch
from mplsoccer import Sbopen
import numpy as np
from matplotlib.colors import LinearSegmentedColormap
from urllib.request import urlopen
from mplsoccer import PyPizza, add_image, FontManager
import seaborn as sns
import plotly.graph_objects as go
import re, json



In [35]:
!pip install -r requirements.txt

  Using cached html5lib-1.1-py2.py3-none-any.whl.metadata (16 kB)
  Using cached lxml-6.1.3-cp313-cp313-win_amd64.whl.metadata (3.4 kB)
  Using cached webencodings-0.6.1-py3-none-any.whl.metadata (3.4 kB)
Using cached html5lib-1.1-py2.py3-none-any.whl (112 kB)
Using cached lxml-6.1.3-cp313-cp313-win_amd64.whl (4.0 MB)
Using cached webencodings-0.6.1-py3-none-any.whl (8.7 kB)

   ------------- -------------------------- 1/3 [lxml]
   ------------- -------------------------- 1/3 [lxml]
   -------------------------- ------------- 2/3 [html5lib]
   -------------------------- ------------- 2/3 [html5lib]
   -------------------------- ------------- 2/3 [html5lib]
   ---------------------------------------- 3/3 [html5lib]



In [37]:
def get_page(league: str, season: int) -> dict:
    """
    Fetch Understat league data and return teams dictionary.

    Parameters
    ----------
    league : str
        League code, e.g. 'EPL', 'LaLiga', 'Bundesliga'
    season : int
        Season start year, e.g. 2025

    Returns
    -------
    dict
        data['teams']
    """

    url = f"https://understat.com/getLeagueData/{league}/{season}"

    headers = {
        "Accept": "application/json, text/javascript, */*; q=0.01",
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/120.0.0.0 Safari/537.36"
        ),
        "X-Requested-With": "XMLHttpRequest",
        "Referer": f"https://understat.com/league/{league}/{season}",
    }

    response = requests.get(url, headers=headers, timeout=15)
    response.raise_for_status()

    data = response.json()
    return data['dates']

In [38]:
data = get_page('EPL', 2026)

In [39]:
def get_matches_for_week(league, year):
    data = get_page(league, year)
    matches = []

    for match in data:
        matches.append({
            'home_team': match['h']['title'],
            'away_team': match['a']['title'],
            'home_goals': match['goals']['h'],
            'away_goals': match['goals']['a'],
            'home_xG': match['xG']['h'],
            'away_xG': match['xG']['a'],
            'datetime': match['datetime']
        })

    df = pd.DataFrame(matches)
    df = df[df['home_goals'].notna()].reset_index(drop=True)

    df['home_xG'] = df['home_xG'].astype(float).round(2)
    df['away_xG'] = df['away_xG'].astype(float).round(2)
    
    return df

In [40]:
df = get_matches_for_week('EPL', 2026)

In [41]:
df

,home_team,away_team,home_goals,away_goals,home_xG,away_xG,datetime
0,Arsenal,Coventry,3,0,1.85,0.56,2026-08-21 19:00:00
1,Hull,Manchester United,2,0,1.50,1.78,2026-08-22 11:30:00
2,Everton,Crystal Palace,2,0,1.45,2.23,2026-08-22 14:00:00
3,Ipswich,Sunderland,2,1,1.58,1.18,2026-08-22 14:00:00
4,Nottingham Forest,Leeds,0,1,0.66,0.47,2026-08-22 14:00:00
5,Brentford,Tottenham,3,0,4.04,0.76,2026-08-22 16:30:00
6,Brighton,Aston Villa,4,0,4.00,0.28,2026-08-23 13:00:00
7,Manchester City,Bournemouth,2,1,2.47,0.71,2026-08-23 13:00:00
8,Newcastle United,Liverpool,2,2,1.59,3.13,2026-08-23 15:30:00
9,Fulham,Chelsea,2,3,1.42,2.58,2026-08-24 19:00:00
